In [23]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os

for dirname, _, filenames in os.walk('/archive/credit_score_copy.csv'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import warnings
warnings.filterwarnings('ignore')

In [24]:
file_path = "C:/Tanu academics/HACKS/pennywiser/archive/credit_score.csv"
df = pd.read_csv(file_path)
df.head()

,CUST_ID,INCOME,SAVINGS,DEBT,R_SAVINGS_INCOME,R_DEBT_INCOME,R_DEBT_SAVINGS,T_CLOTHING_12,T_CLOTHING_6,R_CLOTHING,...,R_EXPENDITURE_SAVINGS,R_EXPENDITURE_DEBT,CAT_GAMBLING,CAT_DEBT,CAT_CREDIT_CARD,CAT_MORTGAGE,CAT_SAVINGS_ACCOUNT,CAT_DEPENDENTS,CREDIT_SCORE,DEFAULT
0,C02COQEVYU,33269,0,532304,0.0000,16.0000,1.2000,1889,945,0.5003,...,0.0000,0.0625,High,1,0,0,0,0,444,1
1,C02OZKC0ZF,77158,91187,315648,1.1818,4.0909,3.4615,5818,111,0.0191,...,0.7692,0.2222,No,1,0,0,1,0,625,0
2,C03FHP2D0A,30917,21642,534864,0.7000,17.3000,24.7142,1157,860,0.7433,...,1.4286,0.0578,High,1,0,0,1,0,469,1
3,C03PVPPHOY,80657,64526,629125,0.8000,7.8000,9.7499,6857,3686,0.5376,...,1.2500,0.1282,High,1,0,0,1,0,559,0
4,C04J69MUX0,149971,1172498,2399531,7.8182,16.0000,2.0465,1978,322,0.1628,...,0.1163,0.0568,High,1,1,1,1,1,473,0


# Preprocessing Data

In [25]:
# drop the customer column from dataset
df = df.drop('CUST_ID', axis = 1)

In [26]:
#replace the values with integer mapping of High = 2, Low =1 and No = 0
gambling = {'High':2,'Low':1,'No':0}
df["CAT_GAMBLING"] = df["CAT_GAMBLING"].map(gambling)
df.head()

,INCOME,SAVINGS,DEBT,R_SAVINGS_INCOME,R_DEBT_INCOME,R_DEBT_SAVINGS,T_CLOTHING_12,T_CLOTHING_6,R_CLOTHING,R_CLOTHING_INCOME,...,R_EXPENDITURE_SAVINGS,R_EXPENDITURE_DEBT,CAT_GAMBLING,CAT_DEBT,CAT_CREDIT_CARD,CAT_MORTGAGE,CAT_SAVINGS_ACCOUNT,CAT_DEPENDENTS,CREDIT_SCORE,DEFAULT
0,33269,0,532304,0.0000,16.0000,1.2000,1889,945,0.5003,0.0568,...,0.0000,0.0625,2,1,0,0,0,0,444,1
1,77158,91187,315648,1.1818,4.0909,3.4615,5818,111,0.0191,0.0754,...,0.7692,0.2222,0,1,0,0,1,0,625,0
2,30917,21642,534864,0.7000,17.3000,24.7142,1157,860,0.7433,0.0374,...,1.4286,0.0578,2,1,0,0,1,0,469,1
3,80657,64526,629125,0.8000,7.8000,9.7499,6857,3686,0.5376,0.0850,...,1.2500,0.1282,2,1,0,0,1,0,559,0
4,149971,1172498,2399531,7.8182,16.0000,2.0465,1978,322,0.1628,0.0132,...,0.1163,0.0568,2,1,1,1,1,1,473,0


In [27]:
# Normalize the data using MinMax Scaler

cols = df.columns
from sklearn.preprocessing import MinMaxScaler
mmax=MinMaxScaler()
df[cols] = mmax.fit_transform(df[cols])

# Goal is to predict "CREDIT_SCORE" feature using the input features. Splitting the set to x and y variables

In [28]:
x = df.drop('CREDIT_SCORE',axis =1)
y = df['CREDIT_SCORE']

In [29]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.metrics import make_scorer, mean_squared_error, mean_absolute_error, r2_score
import joblib


In [30]:
# evaluation helper
def cv_scores(model, X, y, cv=5):
    scoring = {
        "neg_rmse": "neg_root_mean_squared_error",
        "neg_mae": "neg_mean_absolute_error",
        "r2": "r2"
    }
    results = {}
    for name, score in scoring.items():
        vals = cross_val_score(model, X, y, cv=cv, scoring=score, n_jobs=-1)
        # convert negative scores
        if name.startswith("neg_"):
            vals = -vals
        results[name] = (np.mean(vals), np.std(vals))
    return results
# define models to evaluate

In [31]:
# candidate models
models = {
    "LinearRegression": Pipeline([("scaler", StandardScaler()), ("lr", LinearRegression())]),
    "RidgeCV": Pipeline([("scaler", StandardScaler()), ("ridge", RidgeCV(alphas=[0.1, 1.0, 10.0]))]),
    "ElasticNet": Pipeline([("scaler", StandardScaler()), ("en", ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000))]),
    "SVR_RBF": Pipeline([("scaler", StandardScaler()), ("svr", SVR(kernel="rbf", C=1.0, epsilon=0.1))]),
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=200, random_state=42),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=42)
}

# quick CV compare
cv = KFold(n_splits=5, shuffle=True, random_state=42)
print("Cross-validation results (mean ± std):")
summary = {}
for name, model in models.items():
    res = cv_scores(model, x, y, cv=cv)
    summary[name] = res
    print(f"\n{name}")
    print(f" RMSE: {res['neg_rmse'][0]:.4f} ± {res['neg_rmse'][1]:.4f}")
    print(f" MAE:  {res['neg_mae'][0]:.4f} ± {res['neg_mae'][1]:.4f}")
    print(f" R2:   {res['r2'][0]:.4f} ± {res['r2'][1]:.4f}")

Cross-validation results (mean ± std):

LinearRegression
 RMSE: 0.0571 ± 0.0038
 MAE:  0.0418 ± 0.0029
 R2:   0.7967 ± 0.0129

RidgeCV
 RMSE: 0.0553 ± 0.0043
 MAE:  0.0409 ± 0.0030
 R2:   0.8091 ± 0.0161

ElasticNet
 RMSE: 0.0836 ± 0.0067
 MAE:  0.0642 ± 0.0053
 R2:   0.5637 ± 0.0393

SVR_RBF
 RMSE: 0.0728 ± 0.0039
 MAE:  0.0547 ± 0.0022
 R2:   0.6682 ± 0.0326

RandomForest
 RMSE: 0.0587 ± 0.0048
 MAE:  0.0433 ± 0.0034
 R2:   0.7852 ± 0.0199

GradientBoosting
 RMSE: 0.0584 ± 0.0045
 MAE:  0.0422 ± 0.0028
 R2:   0.7870 ± 0.0189

HistGradientBoosting
 RMSE: 0.0582 ± 0.0049
 MAE:  0.0427 ± 0.0035
 R2:   0.7886 ± 0.0210


In [32]:
from joblib import dump, load

# fit the Ridge pipeline before saving
pipeline = models["RidgeCV"]   # this is the Pipeline defined earlier
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

pipeline.fit(X_train, y_train)   # <-- ensure it's fitted
dump(pipeline, 'ridge_pipeline.joblib')
print("Fitted and saved ridge_pipeline.joblib")

Fitted and saved ridge_pipeline.joblib


In [33]:
# Later, you can load it back
from joblib import load

loaded_pipeline = load('ridge_pipeline.joblib')

# Predict for a single row — keep 2D shape
single_row = x.iloc[[0]]   # note the double brackets
y_pred = loaded_pipeline.predict(single_row)
print("Prediction for row 0:", y_pred)
print("Actual value:", y.iloc[0])
converted_value = y_pred[0] * (850 - 300) + 300
print("Converted to original scale (Credit score):", converted_value)



Prediction for row 0: [0.29343949]
Actual value: 0.28800000000000003
Converted to original scale (Credit score): 461.39172133999494


In [34]:
x.iloc[[0]]

,INCOME,SAVINGS,DEBT,R_SAVINGS_INCOME,R_DEBT_INCOME,R_DEBT_SAVINGS,T_CLOTHING_12,T_CLOTHING_6,R_CLOTHING,R_CLOTHING_INCOME,...,R_EXPENDITURE_INCOME,R_EXPENDITURE_SAVINGS,R_EXPENDITURE_DEBT,CAT_GAMBLING,CAT_DEBT,CAT_CREDIT_CARD,CAT_MORTGAGE,CAT_SAVINGS_ACCOUNT,CAT_DEPENDENTS,DEFAULT
0,0.050248,0.0,0.089184,0.0,0.432425,0.004098,0.043671,0.023674,0.472739,0.215062,...,0.249944,0.0,0.006247,1.0,1.0,0.0,0.0,0.0,0.0,1.0
